# 02 — Data Merge & Class Mapping · Banana Ripeness Detection (YOLO26s)

**หน้าที่ของ notebook นี้**
1. ดาวน์โหลด **Mendeley 56c75hvm7r v1** (zip 1.96 GB, YOLO txt, 3 คลาส) → ตรวจ SHA256 → แตกไฟล์ → assert 673 ภาพ
2. ดาวน์โหลด **Roboflow fruit-ripening/banana-ripening-process v1** (YOLO) → assert 7,546 ภาพ
3. Remap class id ของทั้ง 2 ชุดเป็น **unified 4 คลาส** (`0 unripe, 1 semi_ripe, 2 ripe, 3 overripe`) → `data/interim/`
4. สร้าง `reports/manifest_interim.csv` (ข้อมูลรายภาพ สำหรับ nb03 cleaning / nb04 split)
5. **Verify**: class id ∈ 0–3, bbox ∈ [0,1], ไม่มี label หาย/ว่าง, ภาพ↔label ครบคู่
6. เขียน `data/interim/data.yaml` + `reports/02_merge_summary.json` + กราฟ class distribution

**การตัดสินใจที่ใช้ (2026-09-19):** Roboflow มี 6 คลาส — ใช้ 4 ตาม spec และ **ทิ้งภาพที่มี bbox ของ `unripe` / `rotten`** (`drop_mode = "image"`)
เหตุผล: ถ้าลบเฉพาะ bbox แต่เก็บภาพไว้ กล้วยที่ถูกลบ label จะกลายเป็น "background" → โมเดลเรียนรู้ผิด (label noise)
→ จำนวนภาพรวมจะ **น้อยกว่า 8,219 ตาม spec** — notebook จะรายงานตัวเลขจริงและบันทึกลง `configs/project.yaml`

> kernel: **ML Environment (ds69) + YOLO26** · ต้องรัน `01_setup_env.ipynb` ก่อน (ใช้ `configs/project.yaml`)


In [ ]:
# === CELL 1 === โหลด config กลาง + API key + dependencies + seed + สร้าง src/viz.py
# ทำอะไร : อ่าน configs/project.yaml (จาก nb01), โหลด ROBOFLOW_API_KEY จาก .env (ถ้ายังไม่มีให้กรอกแบบซ่อนแล้วบันทึกให้),
#          ติดตั้ง roboflow SDK ถ้าขาด, เขียน src/viz.py (สไตล์กราฟกลาง) และรวมค่าเฉพาะของ nb02 ไว้ใน dict เดียว
# ทำไม   : path/คลาส/seed ต้องมาจากที่เดียว (nb01) และ API key ห้ามอยู่ในโค้ดหรือ notebook ที่จะส่งอาจารย์/ขึ้น git
import os, sys, re, json, shutil, hashlib, zipfile, time, random, subprocess, importlib, importlib.util, getpass
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime
import yaml

PROJECT_ROOT = Path.cwd().resolve()
CFG_PATH = PROJECT_ROOT / "configs" / "project.yaml"
assert CFG_PATH.exists(), f"ไม่พบ {CFG_PATH} → รัน 01_setup_env.ipynb ก่อน"
with open(CFG_PATH, encoding="utf-8") as f:
    CONFIG = yaml.safe_load(f)
DIRS = {k: PROJECT_ROOT / v for k, v in CONFIG["dirs"].items()}
SEED = CONFIG["seed"]
random.seed(SEED)
CLASSES = {int(k): v for k, v in CONFIG["classes"].items()}      # {0:'unripe', 1:'semi_ripe', 2:'ripe', 3:'overripe'}
NC = len(CLASSES)
assert list(CLASSES) == [0, 1, 2, 3], "class index ใน project.yaml ต้องเป็น 0–3"

# --- ค่าเฉพาะ nb02 (รวมไว้ที่เดียว) -----------------------------------------------------------
NB02 = {
    "mendeley": {
        "file_id": "e6fed512-91b4-413e-a7ea-7962ecf01a3c",   # จาก Mendeley public API: BANANA_DATASET.zip
        "sha256": "aa635da1a7e8c4e09056a7a72d8625b2facccbc7f9d9e6a8b3b62c0e3712161e",
        "size_bytes": 1955860652,
        "zip_name": "BANANA_DATASET.zip",
        # Mendeley redirect ไป S3 สาธารณะ (ตรวจ 2026-09-19) — ใช้เป็นทางสำรองถ้าขอ redirect ผ่าน Cloudflare ไม่ได้
        "s3_fallback_url": "https://prod-dcd-datasets-public-files-eu-west-1.s3.eu-west-1.amazonaws.com/4ba3a996-876a-42c1-a8db-d0474f9591cc",
        "class_map": {"unripe": 0, "ripe": 2, "overripe": 3},   # ชื่อคลาสในชุด → unified id (ชุดนี้ไม่มี semi_ripe)
        "fallback_names": ["unripe", "ripe", "overripe"],       # ลำดับ id 0/1/2 ตามหน้า metadata ถ้าใน zip ไม่มี data.yaml
        "prefix": "mdl",
    },
    "roboflow": {
        "format": "yolov8",
        "class_map": CONFIG["datasets"]["roboflow"]["class_map"],   # freshunripe→0 freshripe→1 ripe→2 overripe→3
        "drop_classes": ["unripe", "rotten"],   # ไม่ใช้ 2 คลาสนี้ (ตัดสินใจ 2026-09-19)
        "drop_mode": "image",   # "image" = ทิ้งทั้งภาพที่มี bbox ของคลาสที่ตัด (กัน label noise) | "bbox" = ลบเฉพาะ bbox
        "prefix": "rbf",
    },
    "image_exts": {".jpg", ".jpeg", ".png", ".bmp", ".webp"},
}

# --- dependencies เพิ่มเติมของ nb02 ---------------------------------------------------------
def pip_install(*pkgs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *pkgs]
    print("  $", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"pip install {pkgs} ล้มเหลว:\n{r.stderr[-1500:]}")
    importlib.invalidate_caches()

DEPS = {"requests": "requests", "tqdm": "tqdm", "PIL": "pillow", "pandas": "pandas", "matplotlib": "matplotlib"}
missing = [pkg for mod, pkg in DEPS.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f"[INSTALL] {missing}")
    pip_install(*missing)
print(f"[OK] dependencies ครบ ({len(DEPS)} ตัว)")

# --- ROBOFLOW_API_KEY: อ่านจาก environment → .env → ถ้าไม่มีให้กรอกแบบซ่อน แล้วบันทึกลง .env ให้ ----------------
ENV_PATH = PROJECT_ROOT / ".env"
def load_env_file(path: Path):
    if not path.exists():
        return
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

load_env_file(ENV_PATH)
ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "").strip()
if not ROBOFLOW_API_KEY:
    print("ยังไม่มี ROBOFLOW_API_KEY → วาง Private API Key (app.roboflow.com ▸ Settings ▸ API Keys) ในช่องด้านล่างแล้วกด Enter")
    ROBOFLOW_API_KEY = getpass.getpass("ROBOFLOW_API_KEY: ").strip()
    assert len(ROBOFLOW_API_KEY) >= 10, "key สั้นผิดปกติ — copy มาไม่ครบ?"
    ENV_PATH.write_text(f"ROBOFLOW_API_KEY={ROBOFLOW_API_KEY}\n", encoding="utf-8")
    try:
        os.chmod(ENV_PATH, 0o600)   # อ่านได้เฉพาะเจ้าของไฟล์
    except OSError:
        pass
    print(f"[OK] บันทึกลง {ENV_PATH} แล้ว (ไฟล์ซ่อน — ครั้งหน้าไม่ต้องกรอกอีก)")
print(f"[OK] ROBOFLOW_API_KEY = {ROBOFLOW_API_KEY[:4]}…{ROBOFLOW_API_KEY[-2:]} ({len(ROBOFLOW_API_KEY)} ตัวอักษร)")

# --- src/viz.py: สไตล์กราฟกลาง (เขียนทับทุกครั้งให้เป็นเวอร์ชันล่าสุด) ----------------------------
VIZ_SRC = "\n".join(['# src/viz.py — สไตล์กราฟกลางของโปรเจกต์ (สร้างโดย nb02, ใช้ร่วมกันใน nb02–06)', 'from pathlib import Path', 'import matplotlib.pyplot as plt', '', '# สีประจำ source (categorical slot 1–2 ของ palette ที่ผ่านการตรวจ colorblind-safe) และสีตัวอักษร/เส้นกริด', "SOURCE_COLORS = {'mendeley': '#2a78d6', 'roboflow': '#eb6834'}", "TEXT = {'primary': '#0b0b0b', 'secondary': '#52514e', 'grid': '#e6e5e1'}", '', 'def apply_style():', '    plt.rcParams.update({', "        'figure.dpi': 100, 'savefig.dpi': 300, 'figure.facecolor': 'white', 'axes.facecolor': 'white',", "        'axes.edgecolor': TEXT['secondary'], 'axes.labelcolor': TEXT['primary'], 'text.color': TEXT['primary'],", "        'xtick.color': TEXT['secondary'], 'ytick.color': TEXT['secondary'],", "        'axes.spines.top': False, 'axes.spines.right': False,", "        'axes.grid': True, 'axes.grid.axis': 'y', 'grid.color': TEXT['grid'], 'grid.linewidth': 0.8,", "        'axes.axisbelow': True, 'legend.frameon': False, 'font.size': 10,", '    })', '', 'def class_colors(config):', '    # สีประจำคลาสจาก configs/project.yaml (RGB 0–255 → matplotlib 0–1)', "    return {int(k): tuple(c / 255 for c in v) for k, v in config['class_colors_rgb'].items()}", '', 'def save_fig(fig, figures_dir, name):', '    # บันทึกทุกกราฟเป็น PNG dpi=300 ลง figures/ (ตาม spec ข้อ 11)', '    figures_dir = Path(figures_dir); figures_dir.mkdir(parents=True, exist_ok=True)', "    out = figures_dir / (name if name.endswith('.png') else name + '.png')", "    fig.savefig(out, dpi=300, bbox_inches='tight', facecolor='white')", '    return out']) + "\n"
(DIRS["src"] / "viz.py").write_text(VIZ_SRC, encoding="utf-8")
sys.path.insert(0, str(DIRS["src"]))
import viz
importlib.reload(viz)
viz.apply_style()

RUN_INFO = {"notebook": "02_data_merge_mapping", "started_at": datetime.now().isoformat(timespec="seconds")}
print(f"[OK] PROJECT_ROOT = {PROJECT_ROOT} | seed = {SEED} | classes = {CLASSES}")
print(f"[OK] Roboflow drop_classes = {NB02['roboflow']['drop_classes']} (drop_mode = {NB02['roboflow']['drop_mode']})")


In [ ]:
# === CELL 2 === Mendeley: ดาวน์โหลด (ต่อได้) → ตรวจ SHA256 → แตก zip → สำรวจโครงสร้าง + assert 673 ภาพ
# ทำอะไร : โหลด BANANA_DATASET.zip ผ่าน Mendeley public API แบบ resume ได้, ตรวจ hash ที่ Mendeley ประกาศ, แตกไฟล์
#          แล้ว "ค้นหา" คู่ image/label + data.yaml เอง (ไม่ hardcode โครงสร้างข้างใน zip เพราะยังไม่เคยเห็น)
# ทำไม   : ไฟล์ 1.96 GB หลุดกลางทางได้ → ต้องต่อได้; hash กันไฟล์เสีย; และต้องยืนยันชื่อคลาส/จำนวนจริงก่อน map
import requests
from tqdm import tqdm

MDL = NB02["mendeley"]
MDL_DIR = DIRS["data_raw"] / "mendeley"
MDL_DIR.mkdir(parents=True, exist_ok=True)
zip_path = MDL_DIR / MDL["zip_name"]
extract_dir = MDL_DIR / "extracted"
ok_marker = extract_dir / ".extracted_ok"
ds_id = CONFIG["datasets"]["mendeley"]["id"]
url = f"https://data.mendeley.com/public-files/datasets/{ds_id}/files/{MDL['file_id']}/file_downloaded"

def sha256_of(path: Path, chunk=1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def resolve_download_url(url: str) -> str:
    # Mendeley อยู่หลัง Cloudflare ซึ่งส่งหน้า "Just a moment..." (403) ให้ python-requests แต่ปล่อย curl ผ่าน
    # → ขอแค่ redirect (302 Location) ไปยัง S3 สาธารณะ แล้วค่อยดาวน์โหลดจาก S3 ตรง ๆ (ไม่มี Cloudflare, รองรับ Range)
    try:
        r = requests.get(url, stream=True, timeout=30, allow_redirects=False)
        if r.status_code in (301, 302, 303, 307, 308) and r.headers.get("Location"):
            return r.headers["Location"]
        print(f"  [INFO] requests ได้ HTTP {r.status_code} จาก Mendeley (Cloudflare) → ลองขอ redirect ผ่าน curl")
    except requests.RequestException as e:
        print(f"  [INFO] requests ล้มเหลว: {e} → ลอง curl")
    try:
        out = subprocess.run(["curl", "-sI", "--max-time", "30", url], capture_output=True, text=True, timeout=40).stdout
        m = re.search(r"^location:\s*(\S+)", out, flags=re.I | re.M)
        if m:
            return m.group(1)
    except Exception as e:
        print(f"  [INFO] curl ใช้ไม่ได้: {e}")
    print("  [WARN] ขอ redirect จาก Mendeley ไม่ได้ → ใช้ S3 URL สำรองที่บันทึกไว้ใน NB02['mendeley']['s3_fallback_url']")
    return MDL["s3_fallback_url"]

def download_resumable(url: str, dest: Path, expected_size: int, retries: int = 5) -> None:
    # ดาวน์โหลดลง <dest>.part ด้วย HTTP Range → หลุดแล้วรัน cell ซ้ำจะต่อจากที่ค้าง ไม่เริ่มใหม่
    part = dest.with_suffix(dest.suffix + ".part")
    for attempt in range(1, retries + 1):
        have = part.stat().st_size if part.exists() else 0
        if have >= expected_size:
            break
        headers = {"Range": f"bytes={have}-"} if have else {}
        try:
            with requests.get(url, headers=headers, stream=True, timeout=60, allow_redirects=True) as r:
                if have and r.status_code != 206:      # เซิร์ฟเวอร์ไม่รับ Range → เริ่มใหม่ตั้งแต่ต้น
                    have = 0
                    part.unlink(missing_ok=True)
                r.raise_for_status()
                total = int(r.headers.get("Content-Length", 0)) + have
                assert total == expected_size, f"ขนาดไฟล์บนเซิร์ฟเวอร์ {total:,} ≠ ที่คาด {expected_size:,} → URL/เวอร์ชันของชุดข้อมูลเปลี่ยน"
                with open(part, "ab" if have else "wb") as f, tqdm(total=expected_size, initial=have, unit="B",
                                                                    unit_scale=True, desc=dest.name) as bar:
                    for chunk in r.iter_content(chunk_size=1 << 20):
                        f.write(chunk)
                        bar.update(len(chunk))
        except (requests.RequestException, OSError) as e:
            print(f"  [retry {attempt}/{retries}] {e}")
            time.sleep(5 * attempt)
    got = part.stat().st_size if part.exists() else 0
    assert got == expected_size, (f"ดาวน์โหลดไม่ครบ ({got:,}/{expected_size:,} bytes) → รัน cell นี้ซ้ำเพื่อดาวน์โหลดต่อ "
                                  f"หรือดาวน์โหลดเองจากเบราว์เซอร์ {CONFIG['datasets']['mendeley']['url']} "
                                  f"แล้ววางไฟล์ที่ {dest.relative_to(PROJECT_ROOT)}")
    part.replace(dest)

if zip_path.exists() and zip_path.stat().st_size == MDL["size_bytes"]:
    print(f"[SKIP] มี {zip_path.name} แล้ว ({zip_path.stat().st_size / 1e9:.2f} GB)")
else:
    print(f"[RESOLVE] {url}")
    dl_url = resolve_download_url(url)
    print(f"[DOWNLOAD] {dl_url[:90]}…")
    download_resumable(dl_url, zip_path, MDL["size_bytes"])

if ok_marker.exists():
    print(f"[SKIP] แตกไฟล์แล้วที่ {extract_dir.relative_to(PROJECT_ROOT)}")
else:
    print("[VERIFY] SHA256 (ไฟล์ 1.96 GB ใช้เวลาสักครู่) ...")
    digest = sha256_of(zip_path)
    assert digest == MDL["sha256"], f"SHA256 ไม่ตรง (ได้ {digest[:16]}…) → ไฟล์เสีย ลบ {zip_path.name} แล้วรัน cell นี้ใหม่"
    print("[OK] SHA256 ตรงกับที่ Mendeley ประกาศ")
    print(f"[EXTRACT] → {extract_dir.relative_to(PROJECT_ROOT)}")
    shutil.rmtree(extract_dir, ignore_errors=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extract_dir)
    ok_marker.touch()

# --- helper ใช้ร่วมกับ Roboflow: ค้นหาคู่ image/label และชื่อคลาส ------------------------------------
def is_junk(p: Path) -> bool:
    return "__MACOSX" in p.parts or p.name.startswith("._")

def discover_yolo(root: Path):
    # คืน list ของ {image, label, split}: label = <stem>.txt โดยแทนโฟลเดอร์ "images" ใน path ด้วย "labels"
    # (รองรับทั้ง train/images/x.jpg ↔ train/labels/x.txt ของ Roboflow และ images/train/x.jpg ↔ labels/train/x.txt ของ Mendeley)
    recs = []
    for img in sorted(p for p in root.rglob("*") if p.suffix.lower() in NB02["image_exts"] and not is_junk(p)):
        rel = img.relative_to(root)
        cands = []
        if "images" in [p.lower() for p in rel.parts[:-1]]:
            i = [p.lower() for p in rel.parts].index("images")
            cands.append(root.joinpath(*rel.parts[:i], "labels", *rel.parts[i + 1:-1], img.stem + ".txt"))
        cands.append(img.with_suffix(".txt"))
        lbl = next((c for c in cands if c.exists()), None)
        parts = [p.lower() for p in rel.parts]
        split = next((s for s in ("train", "valid", "val", "test") if s in parts), "unknown")
        recs.append({"image": img, "label": lbl, "split": "valid" if split == "val" else split})
    return recs

def read_names(root: Path):
    # อ่าน names จาก data.yaml ของชุด (รองรับทั้ง list และ dict {id: name})
    for y in sorted(p for p in root.rglob("*.y*ml") if not is_junk(p)):
        try:
            with open(y, encoding="utf-8") as f:
                d = yaml.safe_load(f)
        except Exception:
            continue
        if isinstance(d, dict) and "names" in d:
            names = d["names"]
            if isinstance(names, dict):
                names = [names[k] for k in sorted(names, key=int)]
            return [str(n).strip() for n in names], y
    return None, None

def parse_label(path):
    rows = []
    if path is None or not path.exists():
        return rows
    for ln in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        p = ln.split()
        if len(p) >= 5:
            rows.append((int(float(p[0])), float(p[1]), float(p[2]), float(p[3]), float(p[4])))
    return rows

def print_tree(root: Path, max_depth=2, _prefix="", _depth=0):
    if _depth > max_depth:
        return
    kids = sorted(p for p in root.iterdir() if p.is_dir() and not is_junk(p))
    for i, p in enumerate(kids):
        last = i == len(kids) - 1
        n_img = sum(1 for q in p.rglob("*") if q.suffix.lower() in NB02["image_exts"] and not is_junk(q))
        print(f"{_prefix}{'└── ' if last else '├── '}{p.name}/  ({n_img} images)")
        print_tree(p, max_depth, _prefix + ("    " if last else "│   "), _depth + 1)

# --- สำรวจ Mendeley -------------------------------------------------------------------------------
print(f"\n{extract_dir.name}/")
print_tree(extract_dir)
mdl_names, mdl_yaml = read_names(extract_dir)
if mdl_names is None:
    mdl_names = MDL["fallback_names"]
    print(f"[WARN] ไม่พบ data.yaml ในชุด Mendeley → ใช้ลำดับคลาสตามหน้า metadata: {mdl_names}")
else:
    print(f"[OK] names จาก {mdl_yaml.relative_to(extract_dir)}: {mdl_names}")
mdl_recs = discover_yolo(extract_dir)
mdl_split = Counter(r["split"] for r in mdl_recs)
mdl_nolabel = sum(1 for r in mdl_recs if r["label"] is None)
mdl_inst = Counter()
for r in mdl_recs:
    for cid, *_ in parse_label(r["label"]):
        mdl_inst[mdl_names[cid] if 0 <= cid < len(mdl_names) else f"id{cid}?"] += 1
print(f"[OK] Mendeley: {len(mdl_recs)} ภาพ | split = {dict(mdl_split)} | ไม่มี label file = {mdl_nolabel}")
print(f"     instances ต่อคลาส = {dict(mdl_inst)}")
unknown = set(mdl_names) - set(MDL["class_map"])
assert not unknown, f"Mendeley มีคลาสที่ยังไม่ได้กำหนด mapping: {unknown} → เพิ่มใน NB02['mendeley']['class_map']"
# ชื่อไฟล์ของ Mendeley ขึ้นต้นด้วยชื่อคลาส (unripe_001.jpg ...) → ใช้ตรวจสอบไขว้กับ label (label คือ ground truth; ชื่อไฟล์เป็นแค่ hint)
mismatch = Counter()
for r in mdl_recs:
    hint = r["image"].stem.split("_")[0].lower()
    for cid, *_ in parse_label(r["label"]):
        if hint in mdl_names and mdl_names[cid] != hint:
            mismatch[f"{r['image'].name}: file says {hint}, label says {mdl_names[cid]}"] += 1
if mismatch:
    print(f"[INFO] ชื่อไฟล์กับ label ไม่ตรงกัน {len(mismatch)} ภาพ (ใช้ label เป็นหลัก — บันทึกไว้ให้ nb03 ตรวจด้วยตา):")
    for k in list(mismatch)[:10]:
        print(f"       {k}")
RUN_INFO.setdefault("mendeley_notes", {})["filename_label_mismatch"] = list(mismatch)
exp = CONFIG["datasets"]["mendeley"]["expected_images"]
assert len(mdl_recs) == exp, f"Mendeley ได้ {len(mdl_recs)} ภาพ ไม่ตรง spec {exp} → ตรวจ zip/โครงสร้างก่อนไปต่อ"
RUN_INFO["mendeley"] = {"images": len(mdl_recs), "split": dict(mdl_split), "names": mdl_names,
                        "instances": dict(mdl_inst), "no_label_file": mdl_nolabel}


In [ ]:
# === CELL 3 === Roboflow: ดาวน์โหลด v1 (YOLO) ผ่าน SDK → สำรวจ + assert 7,546 ภาพ + ตรวจชื่อคลาสกับ mapping
# ทำอะไร : ใช้ roboflow SDK โหลด version 1 (ชุดดิบ ไม่ใช่ v2 ที่ augment แล้ว) ลง data/raw/roboflow/ แล้วนับภาพ/instance
# ทำไม   : ต้องยืนยันว่าได้ 7,546 ภาพตาม spec และรู้ครบว่ามีคลาสอะไรบ้าง (6 คลาส) ก่อนตัดสินใจ map/ทิ้ง
RBF_CFG = CONFIG["datasets"]["roboflow"]
RBF = NB02["roboflow"]
RBF_DIR = DIRS["data_raw"] / "roboflow" / f"{RBF_CFG['project']}-v{RBF_CFG['version']}"

if (RBF_DIR / "data.yaml").exists():
    print(f"[SKIP] มี {RBF_DIR.relative_to(PROJECT_ROOT)} แล้ว")
else:
    if importlib.util.find_spec("roboflow") is None:
        print("[INSTALL] roboflow SDK")
        pip_install("roboflow")
    from roboflow import Roboflow
    print(f"[DOWNLOAD] Roboflow {RBF_CFG['workspace']}/{RBF_CFG['project']} v{RBF_CFG['version']} "
          f"({RBF['format']}) → {RBF_DIR.relative_to(PROJECT_ROOT)}")
    try:
        rf = Roboflow(api_key=ROBOFLOW_API_KEY)
        version = rf.workspace(RBF_CFG["workspace"]).project(RBF_CFG["project"]).version(RBF_CFG["version"])
        version.download(RBF["format"], location=str(RBF_DIR), overwrite=True)
    except Exception as e:
        raise RuntimeError(f"ดาวน์โหลด Roboflow ไม่สำเร็จ: {e}\n"
                           f"  → เช็คว่า key ใน {ENV_PATH.name} ถูกต้อง (Private API Key) และเซิร์ฟเวอร์ออกอินเทอร์เน็ตได้\n"
                           f"  → ถ้าแก้ key: ลบไฟล์ .env แล้วรัน CELL 1 ใหม่") from e
    assert (RBF_DIR / "data.yaml").exists(), "ดาวน์โหลดเสร็จแต่ไม่พบ data.yaml — โครงสร้างเปลี่ยน?"

print(f"\n{RBF_DIR.name}/")
print_tree(RBF_DIR, max_depth=1)
rbf_names, rbf_yaml = read_names(RBF_DIR)
assert rbf_names, "ไม่พบ names ใน data.yaml ของ Roboflow"
rbf_recs = discover_yolo(RBF_DIR)
rbf_split = Counter(r["split"] for r in rbf_recs)
rbf_nolabel = sum(1 for r in rbf_recs if r["label"] is None)
rbf_inst, rbf_img_with = Counter(), Counter()
for r in rbf_recs:
    seen = set()
    for cid, *_ in parse_label(r["label"]):
        nm = rbf_names[cid] if 0 <= cid < len(rbf_names) else f"id{cid}?"
        rbf_inst[nm] += 1
        seen.add(nm)
    for nm in seen:
        rbf_img_with[nm] += 1
print(f"[OK] names ({len(rbf_names)}): {rbf_names}")
print(f"[OK] Roboflow: {len(rbf_recs)} ภาพ | split = {dict(rbf_split)} | ไม่มี label file = {rbf_nolabel}")
print("     คลาส            instances   ภาพที่มี   → การจัดการ")
for nm in rbf_names:
    action = (f"map → {RBF['class_map'][nm]} ({CLASSES[RBF['class_map'][nm]]})" if nm in RBF["class_map"]
              else "DROP" if nm in RBF["drop_classes"] else "?? ไม่ได้กำหนด")
    print(f"     {nm:<15} {rbf_inst[nm]:>9,} {rbf_img_with[nm]:>9,}   → {action}")
unknown = set(rbf_names) - set(RBF["class_map"]) - set(RBF["drop_classes"])
assert not unknown, f"Roboflow มีคลาสที่ไม่ได้กำหนดว่า map หรือ drop: {unknown}"
exp = RBF_CFG["expected_images"]
assert len(rbf_recs) == exp, (f"Roboflow ได้ {len(rbf_recs)} ภาพ ไม่ตรง spec {exp} → ตรวจว่าโหลด version 1 (raw) จริง "
                              f"ไม่ใช่ version ที่ augment แล้ว")
RUN_INFO["roboflow"] = {"images": len(rbf_recs), "split": dict(rbf_split), "names": rbf_names,
                        "instances": dict(rbf_inst), "images_with_class": dict(rbf_img_with), "no_label_file": rbf_nolabel}


In [ ]:
# === CELL 4 === Remap class id → unified 4 คลาส แล้วรวมไฟล์ลง data/interim/{images,labels}
# ทำอะไร : อ่าน label ทุกไฟล์ → แปลง id ผ่านชื่อคลาส (ไม่ใช่ผ่านตัวเลข เพราะลำดับ id ของแต่ละชุดไม่เหมือนกัน)
#          → เขียน label ใหม่ + copy ภาพ ตั้งชื่อ <prefix>_<ชื่อเดิม> กันชนกัน → เก็บ metadata รายภาพไว้ทำ manifest
# ทำไม   : Roboflow ทิ้งทั้งภาพที่มี unripe/rotten (drop_mode=image) เพราะการลบ bbox แต่เก็บภาพ = สอนโมเดลว่ากล้วยนั้นเป็น background
from PIL import Image, ImageOps

IM_OUT = DIRS["data_interim"] / "images"
LB_OUT = DIRS["data_interim"] / "labels"
for d in (IM_OUT, LB_OUT):          # เริ่มใหม่ทุกครั้ง (idempotent) ไม่ให้ไฟล์จากรอบก่อนค้าง
    shutil.rmtree(d, ignore_errors=True)
    d.mkdir(parents=True, exist_ok=True)

def group_id_of(stem: str) -> str:
    # กลุ่มภาพต้นทางเดียวกัน (ใช้กัน leakage ใน nb04): Roboflow ตั้งชื่อ <orig>_jpg.rf.<hash32> → ตัดส่วนท้ายออก
    s = re.sub(r"\.rf\.[0-9a-f]{32}$", "", stem, flags=re.I)
    s = re.sub(r"_(jpe?g|png|bmp|webp)$", "", s, flags=re.I)
    return s.lower()

def copy_image(src: Path, dst: Path):
    # copy ภาพลง interim: ถ้ามี EXIF orientation ≠ 1 → หมุน pixel จริงตาม tag แล้วบันทึกใหม่ (ไม่ใส่ tag)
    # ทำไม: label ของ Mendeley ถูกวาดบนภาพ "หลังหมุน" (ตรวจด้วยตา 2026-09-19) — PIL/บาง tool ไม่หมุนให้อัตโนมัติ
    #       ถ้าไม่ทำตรงนี้ bbox จะไปอยู่ผิดที่ในภาพแนวตั้ง 376/673 ภาพ; คืน (raw_w, raw_h, w, h, orientation, transposed)
    with Image.open(src) as im:
        raw_w, raw_h = im.size
        ori = int(im.getexif().get(274, 1))
        if ori == 1:
            shutil.copy2(src, dst)
            return raw_w, raw_h, raw_w, raw_h, ori, False
        fixed = ImageOps.exif_transpose(im)
        if dst.suffix.lower() in (".jpg", ".jpeg"):
            fixed.save(dst, quality=95, subsampling=0)
        else:
            fixed.save(dst)
        return raw_w, raw_h, fixed.width, fixed.height, ori, True

manifest, drop_log, used = [], Counter(), set()

def merge_source(recs, src, names, class_map, drop_classes=(), drop_mode="image", prefix="x"):
    n_ok = 0
    for r in tqdm(recs, desc=f"merge {src}"):
        if r["label"] is None:
            drop_log[f"{src}:no_label_file"] += 1
            continue
        rows = parse_label(r["label"])
        if not rows:
            drop_log[f"{src}:empty_label"] += 1
            continue
        new_rows, n_drop, n_clip, bad = [], 0, 0, None
        for cid, x, y, w, h in rows:
            if not 0 <= cid < len(names):
                bad = f"id{cid}"
                break
            nm = names[cid]
            if nm in drop_classes:
                n_drop += 1
                continue
            if nm not in class_map:
                bad = nm
                break
            # clip bbox ให้อยู่ในภาพ: Mendeley มีกล้วยที่ล้นขอบเฟรม (w/h > 1 หรือขอบ < 0) — ultralytics จะ "ทิ้งทั้งภาพ" ถ้าค่าใดเกิน [0,1]
            x1, y1, x2, y2 = max(0.0, x - w / 2), max(0.0, y - h / 2), min(1.0, x + w / 2), min(1.0, y + h / 2)
            if x2 - x1 < 1e-4 or y2 - y1 < 1e-4:
                drop_log[f"{src}:bbox_degenerate_after_clip"] += 1
                continue
            if abs(x1 - (x - w / 2)) > 1e-6 or abs(y1 - (y - h / 2)) > 1e-6 or abs(x2 - (x + w / 2)) > 1e-6 or abs(y2 - (y + h / 2)) > 1e-6:
                n_clip += 1
            new_rows.append((class_map[nm], (x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1))
        if bad is not None:
            drop_log[f"{src}:unknown_class:{bad}"] += 1
            continue
        if n_drop and drop_mode == "image":
            drop_log[f"{src}:image_has_dropped_class"] += 1
            continue
        if n_drop:
            drop_log[f"{src}:bbox_dropped"] += n_drop
        if n_clip:
            drop_log[f"{src}:bbox_clipped_to_image (แก้แล้ว ไม่ได้ทิ้ง)"] += n_clip
        if not new_rows:
            drop_log[f"{src}:empty_after_drop"] += 1
            continue
        stem = base = f"{prefix}_{r['image'].stem}"
        k = 1
        while stem in used:                      # กันชื่อซ้ำ (เช่นชื่อเดียวกันคนละ split)
            k += 1
            stem = f"{base}_{k}"
        used.add(stem)
        dst_img = IM_OUT / (stem + r["image"].suffix.lower())
        dst_lbl = LB_OUT / (stem + ".txt")
        raw_w, raw_h, W, H, ori, transposed = copy_image(r["image"], dst_img)
        dst_lbl.write_text("".join(f"{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n" for c, x, y, w, h in new_rows), encoding="utf-8")
        cnt = Counter(c for c, *_ in new_rows)
        area = defaultdict(float)
        for c, x, y, w, h in new_rows:
            area[c] += w * h
        dominant = max(cnt, key=lambda c: (cnt[c], area[c]))      # คลาสหลักของภาพ = มากสุด (เสมอกัน → พื้นที่รวมมากสุด)
        manifest.append({"file": dst_img.name, "label": dst_lbl.name, "source": src, "orig_split": r["split"],
                         "orig_name": r["image"].name, "group_id": group_id_of(r["image"].stem),
                         "width": W, "height": H, "raw_width": raw_w, "raw_height": raw_h,
                         "exif_orientation": ori, "exif_transposed": transposed,
                         "n_boxes": len(new_rows), "classes": ";".join(str(c) for c in sorted(cnt)),
                         "dominant_class": dominant, "dominant_name": CLASSES[dominant],
                         **{f"n_{CLASSES[c]}": cnt.get(c, 0) for c in CLASSES},
                         "boxes_dropped": n_drop, "boxes_clipped": n_clip})
        n_ok += 1
    return n_ok

n_mdl = merge_source(mdl_recs, "mendeley", mdl_names, MDL["class_map"], prefix=MDL["prefix"])
n_rbf = merge_source(rbf_recs, "roboflow", rbf_names, RBF["class_map"], RBF["drop_classes"], RBF["drop_mode"], RBF["prefix"])

print(f"\n[OK] merged → {IM_OUT.relative_to(PROJECT_ROOT)}: mendeley {n_mdl:,} + roboflow {n_rbf:,} = {n_mdl + n_rbf:,} ภาพ")
print("     สาเหตุที่ภาพ/bbox ถูกตัดออกหรือแก้ไข:")
for k, v in sorted(drop_log.items()):
    print(f"       {k:<40} {v:>6,}")
if n_mdl != len(mdl_recs):
    print(f"[SPEC MISMATCH] Mendeley ใช้ได้ {n_mdl} จาก {len(mdl_recs)} ภาพ (ดูสาเหตุด้านบน)")
if n_rbf != len(rbf_recs):
    print(f"[SPEC MISMATCH] Roboflow ใช้ได้ {n_rbf:,} จาก {len(rbf_recs):,} ภาพ — ตามการตัดสินใจทิ้ง {RBF['drop_classes']} (drop_mode={RBF['drop_mode']})")
assert n_mdl + n_rbf == len(list(IM_OUT.iterdir())) == len(list(LB_OUT.iterdir())), "จำนวนไฟล์ใน interim ไม่ตรงกับที่นับได้"
RUN_INFO["merge"] = {"mendeley_used": n_mdl, "roboflow_used": n_rbf, "total": n_mdl + n_rbf, "drop_log": dict(drop_log)}


In [ ]:
# === CELL 5 === manifest_interim.csv + ตารางสรุป (ต่อ source / split / คลาส) + ข้อสังเกตเรื่องขนาดภาพ & EXIF
# ทำอะไร : เก็บ metadata รายภาพเป็น CSV (nb03 ใช้ทำ cleaning, nb04 ใช้ทำ stratified split + group กัน leakage)
# ทำไม   : ทุกขั้นตอนถัดไปต้องอ้างอิงตารางเดียวกัน ไม่ต้องสแกนไฟล์ซ้ำ และตัวเลขในเล่มรายงานต้องมาจากไฟล์นี้
import pandas as pd
pd.set_option("display.width", 160)

df = pd.DataFrame(manifest)
csv_path = DIRS["reports"] / "manifest_interim.csv"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"[OK] {csv_path.relative_to(PROJECT_ROOT)}: {len(df):,} แถว × {df.shape[1]} คอลัมน์")

print("\n— ภาพต่อ source × split เดิม —")
print(pd.crosstab(df["source"], df["orig_split"], margins=True))

print("\n— ภาพต่อ dominant class (image-level) —")
print(pd.crosstab(df["source"], df["dominant_name"], margins=True).reindex(columns=[*CLASSES.values(), "All"], fill_value=0))

inst_cols = [f"n_{v}" for v in CLASSES.values()]
inst = df.groupby("source")[inst_cols].sum()
inst.loc["All"] = inst.sum()
inst["total"] = inst.sum(axis=1)
print("\n— instances (bbox) ต่อคลาส (instance-level) —")
print(inst)

print("\n— ขนาดภาพที่พบบ่อย (w×h) —")
for src, g in df.groupby("source"):
    top = (g["width"].astype(str) + "×" + g["height"].astype(str)).value_counts().head(4)
    print(f"  {src:<9}: " + ", ".join(f"{k} ({v})" for k, v in top.items()))
n_exif = int((df["exif_orientation"].fillna(1) != 1).sum())
n_fixed = int(df["exif_transposed"].sum())
print(f"\n[INFO] ภาพที่มี EXIF orientation ≠ 1 (กล้องใส่ tag หมุน): {n_exif} ภาพ → หมุน pixel จริงแล้วตอน copy {n_fixed} ภาพ "
      f"(width/height ใน manifest = ขนาดหลังหมุน, raw_width/raw_height = ขนาดเดิม)")
assert n_exif == n_fixed, "มีภาพที่ติด EXIF orientation แต่ไม่ได้หมุน"
RUN_INFO["exif_rotated_images"] = n_exif


In [ ]:
# === CELL 6 === Verify label ทั้งหมด + กราฟ class distribution (image-level & instance-level ต่อ source)
# ทำอะไร : สแกน data/interim/labels ทุกไฟล์: class id ∈ 0–3, ค่า x y w h ∈ [0,1], w/h > 0, ภาพ↔label ครบคู่, ไม่มีไฟล์ว่าง
# ทำไม   : spec ข้อ 4 บังคับตรวจก่อนใช้ — label ผิด 1 ไฟล์ทำให้ ultralytics ข้ามภาพเงียบ ๆ หรือเทรนผิดคลาส
import numpy as np
import matplotlib.pyplot as plt

lbl_files = sorted(LB_OUT.glob("*.txt"))
img_files = sorted(p for p in IM_OUT.iterdir() if p.suffix.lower() in NB02["image_exts"])
assert {p.stem for p in img_files} == {p.stem for p in lbl_files}, "ภาพกับ label ไม่ครบคู่กัน"
assert len(lbl_files) == len(df), "จำนวน label ไม่ตรงกับ manifest"

issues, n_boxes, EPS = Counter(), 0, 1e-6
for lf in lbl_files:
    rows = parse_label(lf)
    if not rows:
        issues["empty_label"] += 1
    for c, x, y, w, h in rows:
        n_boxes += 1
        if c not in CLASSES:
            issues["class_out_of_range"] += 1
        if any(v < -EPS or v > 1 + EPS for v in (x, y, w, h)):
            issues["coord_out_of_[0,1]"] += 1
        if w <= 0 or h <= 0:
            issues["nonpositive_wh"] += 1
        if x - w / 2 < -0.01 or x + w / 2 > 1.01 or y - h / 2 < -0.01 or y + h / 2 > 1.01:
            issues["box_extends_past_edge(>1%)"] += 1     # รายงานอย่างเดียว (ultralytics จะ clip ให้)
print(f"[CHECK] labels {len(lbl_files):,} ไฟล์ | boxes {n_boxes:,} | issues = {dict(issues) or 'none'}")
hard = {k: v for k, v in issues.items() if k in ("empty_label", "class_out_of_range", "coord_out_of_[0,1]", "nonpositive_wh")}
assert not hard, f"พบ label ผิดพลาดร้ายแรง: {hard} → ต้องแก้ก่อนไป nb03"
print("[OK] ผ่าน: class id ∈ 0–3, bbox ∈ [0,1], ไม่มี label ว่าง, ภาพ↔label ครบคู่")

# --- กราฟ: 2 panel (instance-level / image-level) แท่งแยกตาม source, สีตาม source, ตัวเลขกำกับบนแท่ง ---
sources = ["mendeley", "roboflow"]
names = list(CLASSES.values())
inst_by = {s: [int(df.loc[df.source == s, f"n_{n}"].sum()) for n in names] for s in sources}
img_by = {s: [int(((df.source == s) & (df.dominant_class == c)).sum()) for c in CLASSES] for s in sources}

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
x = np.arange(len(names))
bw = 0.36
for ax, data, title in ((axes[0], inst_by, "Instances (bounding boxes) per class"),
                        (axes[1], img_by, "Images per dominant class")):
    for i, s in enumerate(sources):
        bars = ax.bar(x + (i - 0.5) * bw, data[s], width=bw - 0.02, color=viz.SOURCE_COLORS[s], label=s.capitalize())
        for b, v in zip(bars, data[s]):
            ax.text(b.get_x() + b.get_width() / 2, b.get_height(), f"{v:,}", ha="center", va="bottom",
                    fontsize=8, color=viz.TEXT["secondary"])
    ax.set_xticks(x, names)
    ax.set_title(title, loc="left", fontsize=11)
    ax.set_ylabel("count")
    ax.margins(y=0.15)
handles, labels = axes[0].get_legend_handles_labels()
fig.suptitle(f"Merged dataset after class remap — {len(df):,} images, {n_boxes:,} boxes (before cleaning)", x=0.01, ha="left", fontsize=12)
fig.tight_layout(rect=(0, 0, 1, 0.94))
fig.legend(handles, labels, loc="upper right", ncol=2, bbox_to_anchor=(0.99, 0.995))   # legend อยู่แถวเดียวกับ title → ไม่ทับตัวเลขบนแท่ง
out = viz.save_fig(fig, DIRS["figures"], "02_class_distribution.png")
plt.show()
print(f"[OK] figure → {out.relative_to(PROJECT_ROOT)}")
RUN_INFO["verify"] = {"labels": len(lbl_files), "boxes": n_boxes, "issues": dict(issues)}


In [ ]:
# === CELL 7 === เขียน data/interim/data.yaml + reports/02_merge_summary.json + บันทึกตัวเลขจริงลง project.yaml
# ทำอะไร : data.yaml ชั่วคราว (4 คลาส, ยังไม่ split) + สรุปทุกตัวเลขของ nb02 + คำนวณ split เป้าหมาย 80/10/10 จากจำนวนจริง
# ทำไม   : จำนวนภาพจริงต่างจาก spec (ทิ้ง unripe/rotten) → nb04 ต้องใช้ตัวเลขจริง ไม่ใช่ 6575/822/822 ที่ spec เขียนไว้
interim_yaml = {
    "path": str(DIRS["data_interim"]),
    "train": "images", "val": "images",     # placeholder — nb04 จะเขียน datasets/banana/data.yaml ตัวจริงหลัง split
    "nc": NC,
    "names": {k: v for k, v in CLASSES.items()},
    "note": "interim merged set (unsplit, uncleaned) — generated by 02_data_merge_mapping.ipynb",
}
interim_yaml_path = DIRS["data_interim"] / "data.yaml"
with open(interim_yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(interim_yaml, f, allow_unicode=True, sort_keys=False)

N = len(df)
ratio = CONFIG["datasets"]["split_ratio"]
proj_train, proj_valid = round(N * ratio["train"]), round(N * ratio["valid"])
projected = {"train": proj_train, "valid": proj_valid, "test": N - proj_train - proj_valid}
spec_total, spec_split = CONFIG["datasets"]["expected_total"], CONFIG["datasets"]["expected_split"]

print("— SPEC vs ACTUAL —")
print(f"  total images : spec {spec_total:,}  → actual {N:,}  ({N - spec_total:+,})")
for k in ("train", "valid", "test"):
    print(f"  {k:<13}: spec {spec_split[k]:,}  → projected {projected[k]:,}  (ก่อน cleaning ใน nb03 — ตัวเลขสุดท้ายอยู่ที่ nb04)")
print(f"  instances    : {n_boxes:,} boxes | ต่อคลาส = " + ", ".join(f"{n} {int(df[f'n_{n}'].sum()):,}" for n in names))

CONFIG["datasets"]["actual_after_merge"] = {
    "total": N, "mendeley": int((df.source == "mendeley").sum()), "roboflow": int((df.source == "roboflow").sum()),
    "roboflow_dropped_classes": RBF["drop_classes"], "roboflow_drop_mode": RBF["drop_mode"],
    "images_dropped": {k: v for k, v in drop_log.items()},
    "projected_split": projected, "updated_by": "02_data_merge_mapping.ipynb",
}
CONFIG["datasets"]["mendeley"]["class_map"] = MDL["class_map"]
with open(CFG_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(CONFIG, f, allow_unicode=True, sort_keys=False)

RUN_INFO.update({"spec_total": spec_total, "actual_total": N, "projected_split": projected,
                 "finished_at": datetime.now().isoformat(timespec="seconds")})
summary_path = DIRS["reports"] / "02_merge_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(RUN_INFO, f, ensure_ascii=False, indent=2, default=str)

print(f"\n[OK] {interim_yaml_path.relative_to(PROJECT_ROOT)}")
print(f"[OK] {summary_path.relative_to(PROJECT_ROOT)}")
print(f"[OK] อัปเดต {CFG_PATH.relative_to(PROJECT_ROOT)} → datasets.actual_after_merge")
print("\n=== SUMMARY · Notebook 02 ===")
print(f"  Mendeley {RUN_INFO['mendeley']['images']} → ใช้ {RUN_INFO['merge']['mendeley_used']} | "
      f"Roboflow {RUN_INFO['roboflow']['images']:,} → ใช้ {RUN_INFO['merge']['roboflow_used']:,} | รวม {N:,} ภาพ, {n_boxes:,} boxes")
print(f"  ไฟล์: data/interim/{{images,labels}} | reports/manifest_interim.csv | figures/02_class_distribution.png")
print(f"  ถัดไป: 03_data_cleaning_eda.ipynb")


## ผลลัพธ์ที่ควรเห็น
- CELL 2: `[OK] SHA256 ตรง`, โครงสร้างโฟลเดอร์ของ Mendeley, `names = [...]` 3 คลาส, **673 ภาพ**
- CELL 3: ตารางคลาส Roboflow 6 แถว (4 map / 2 DROP), **7,546 ภาพ**
- CELL 4: จำนวนภาพที่ใช้ได้ต่อ source + สาเหตุที่ตัด (`roboflow:image_has_dropped_class` คือส่วนที่หายไปตามการตัดสินใจ)
- CELL 6: `issues = none` (หรือมีแค่ `box_extends_past_edge` ซึ่งไม่ร้ายแรง) + กราฟ 2 panel
- CELL 7: ตาราง SPEC vs ACTUAL — **ตัวเลขจริงจะน้อยกว่า 8,219** ใช้ตัวนี้ในเล่มรายงาน

## ปัญหาที่อาจเจอ + วิธีแก้
| อาการ | สาเหตุ | วิธีแก้ |
|---|---|---|
| CELL 2 ดาวน์โหลดหลุด / `ดาวน์โหลดไม่ครบ` | เน็ตหลุด, ไฟล์ 1.96 GB | รัน CELL 2 ซ้ำ — ต่อจากที่ค้างอัตโนมัติ (`.part`) |
| CELL 2 `ดาวน์โหลดไม่ครบ (0/…)` ทุกครั้ง | เซิร์ฟเวอร์ออกอินเทอร์เน็ตไม่ได้ / Mendeley เปลี่ยน URL | ดาวน์โหลด `BANANA_DATASET.zip` จากเบราว์เซอร์ที่ https://data.mendeley.com/datasets/56c75hvm7r/1 แล้ว upload ไปที่ `data/raw/mendeley/` แล้วรัน CELL 2 ใหม่ (จะข้ามการดาวน์โหลดและตรวจ SHA256 ให้) |
| `SHA256 ไม่ตรง` | ไฟล์เสีย | ลบ `data/raw/mendeley/BANANA_DATASET.zip` แล้วรัน CELL 2 ใหม่ |
| CELL 3 `ดาวน์โหลด Roboflow ไม่สำเร็จ` | key ผิด / ไม่ใช่ Private key / เซิร์ฟเวอร์ปิดเน็ต | ลบไฟล์ `.env` → รัน CELL 1 ใหม่แล้วกรอก key ใหม่ |
| CELL 3 `Roboflow ได้ N ภาพ ไม่ตรง spec 7546` | โหลดผิด version | เช็ค `datasets.roboflow.version` ใน project.yaml ต้องเป็น 1 |
| CELL 2/3 `มีคลาสที่ยังไม่ได้กำหนด mapping` | ชื่อคลาสจริงต่างจากที่คาด | copy ชื่อคลาสที่พิมพ์ออกมาส่งให้ผู้พัฒนา — ห้ามเดา mapping เอง |
| CELL 2 `[INFO] ชื่อไฟล์กับ label ไม่ตรงกัน` | ผู้ annotate ตั้งชื่อไฟล์ตามคลาสแต่ label ต่างออกไป (พบ 3 ภาพ overripe_* ที่ label เป็น ripe) | ไม่ต้องแก้ — ใช้ label เป็นหลัก, nb03 จะแสดงภาพให้ดูด้วยตา |
| CELL 4 `bbox_clipped_to_image` มีจำนวน | Mendeley มีกล้วยที่ล้นขอบภาพ (annotator ลาก bbox เลยขอบ) | ปกติ — bbox ถูก clip ให้อยู่ใน [0,1] แล้ว (ถ้าไม่ทำ ultralytics จะทิ้งทั้งภาพเงียบ ๆ) |
| CELL 6 `พบ label ผิดพลาดร้ายแรง` | ข้อมูลต้นทางมี label เสีย | ส่ง `issues` ที่พิมพ์มา — จะเพิ่มขั้น auto-fix ใน nb03 |
